In [43]:
import torch
from torch.utils.data import Dataset, DataLoader

In [90]:
# Create Toy Data
X_train = torch.tensor(
    [
        [-1.2, 3.1, -2.1],
        [-0.9, 2.9, -1.2],
        [-0.5, 2.6, -1.4],
        [2.3, -1.1, 2.1],
        [2.7, -1.5, 1.5],
    ]
)
y_train = torch.tensor([0, 0, 0, 1, 1])

# Create Test Data
X_test = torch.tensor(
    [
        [-0.8, 2.8, -2.5],
        [2.6, -1.6, 2.1],
    ]
)
y_test = torch.tensor([0, 1])

X_train.shape, y_train.shape, X_test.shape, y_test.shape

(torch.Size([5, 3]), torch.Size([5]), torch.Size([2, 3]), torch.Size([2]))

In [91]:
class ToyDataset(Dataset):
    def __init__(self, features, labels):
        self.features = features
        self.labels = labels

    def __getitem__(self, key):
        return self.features[0], self.labels[0]

    def __len__(self):
        return self.features.shape[0]


train_ds = ToyDataset(X_train, y_train)
test_ds = ToyDataset(X_test, y_test)

len(train_ds), len(test_ds)

(5, 2)

In [92]:
train_dl = DataLoader(
    train_ds, batch_size=2, shuffle=True, num_workers=0, drop_last=True
)
test_dl = DataLoader(test_ds, batch_size=2, shuffle=False, num_workers=0)

In [93]:
for idx, (x, y) in enumerate(train_dl):
    print(f"{idx=} {x=} {y=}")

idx=0 x=tensor([[-1.2000,  3.1000, -2.1000],
        [-1.2000,  3.1000, -2.1000]]) y=tensor([0, 0])
idx=1 x=tensor([[-1.2000,  3.1000, -2.1000],
        [-1.2000,  3.1000, -2.1000]]) y=tensor([0, 0])


In [97]:
# Model


class ToyModel(torch.nn.Module):
    def __init__(self, ip_dims, out_dims):
        super().__init__()
        self.layers = torch.nn.Sequential(
            torch.nn.Linear(ip_dims, 8),
            torch.nn.ReLU(),
            torch.nn.Linear(8, 5),
            torch.nn.ReLU(),
            torch.nn.Linear(5, out_dims),
        )

    def forward(self, inps):
        return self.layers(inps)


toy_model = ToyModel(3, 2)
toy_model

ToyModel(
  (layers): Sequential(
    (0): Linear(in_features=3, out_features=8, bias=True)
    (1): ReLU()
    (2): Linear(in_features=8, out_features=5, bias=True)
    (3): ReLU()
    (4): Linear(in_features=5, out_features=2, bias=True)
  )
)

In [98]:
# Train Loop
num_epochs = 5
# model = toy_model
optim = torch.optim.SGD(toy_model.parameters(), lr=0.05)

In [100]:
for epoch in range(num_epochs):
    toy_model.train()
    for idx, (x, y) in enumerate(train_dl):
        outs = toy_model(x)

        loss = torch.nn.functional.cross_entropy(outs, y)

        optim.zero_grad()
        loss.backward()
        optim.step()

    print(f" {epoch=} | {idx=} | loss={loss:.2f}")  # | {y=} | {outs=}")

 epoch=0 | idx=1 | loss=0.14
 epoch=1 | idx=1 | loss=0.10
 epoch=2 | idx=1 | loss=0.08
 epoch=3 | idx=1 | loss=0.06
 epoch=4 | idx=1 | loss=0.05


In [103]:
toy_model.eval()
for x, y in test_dl:
    logits = toy_model(x)
    print(f"{y=} -- {torch.argmax(logits, dim=1)=}")

y=tensor([0, 0]) -- torch.argmax(logits, dim=1)=tensor([0, 0])
